# Notebook 03b: SRS Scores (XuetangX)

**Purpose:** Compute Session Reliability Score (SRS) per session and attach to pairs.

**SRS formula (canonical):**
```
intensity   = min(n_events / CAP_INTENSITY, 1.0)   # CAP = 95th pct of TRAIN sessions
extent      = min(duration_sec / CAP_EXTENT, 1.0)  # CAP = 95th pct of TRAIN sessions
composition = min(n_unique_action_types / 6, 1.0)
SRS         = (intensity + extent + composition) / 3
```

**Inputs:**
- `data/processed/xuetangx/sessions/sessions.parquet`
- `data/processed/xuetangx/pairs/pairs.parquet`
- `data/processed/xuetangx/user_splits/users_train.json` (for CAP computation — no leakage)

**Outputs:**
- `data/processed/xuetangx/pairs_srs/pairs.parquet` (pairs + srs column)
- `data/processed/xuetangx/pairs_srs/session_srs.parquet`

In [1]:
# [CELL 03b-00] Bootstrap

import json, time, uuid, hashlib, math
from pathlib import Path
from datetime import datetime
from typing import Any
import numpy as np
import pandas as pd

def find_repo_root(start):
    for p in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (p / 'PROJECT_STATE.md').exists(): return p
    raise RuntimeError('PROJECT_STATE.md not found')

REPO_ROOT = find_repo_root(Path.cwd())
PATHS = {
    'META_REGISTRY':  REPO_ROOT / 'meta.json',
    'DATA_PROCESSED': REPO_ROOT / 'data' / 'processed',
    'REPORTS':        REPO_ROOT / 'reports',
}

def cell_start(cid, title, **kw):
    t = time.time()
    print(f'\n[{cid}] {title}')
    for k,v in kw.items(): print(f'[{cid}] {k}={v}')
    return t

def cell_end(cid, t0, **kw):
    for k,v in kw.items(): print(f'[{cid}] {k}={v}')
    print(f'[{cid}] elapsed={time.time()-t0:.2f}s  done')

def write_json_atomic(path, obj, indent=2):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f'.tmp_{uuid.uuid4().hex}')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding='utf-8')
    tmp.replace(path)

def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def sha256_file(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        while chunk := f.read(1<<20): h.update(chunk)
    return h.hexdigest()

print(f'[CELL 03b-00] REPO_ROOT={REPO_ROOT}  done')

[CELL 03b-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta  done


In [2]:
# [CELL 03b-01] Seed + run config + paths

import random
GLOBAL_SEED = 20260107
random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED)

NOTEBOOK_NAME = '03b_srs_scores_xuetangx'
DATASET       = 'xuetangx'
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ID  = uuid.uuid4().hex

OUT_DIR = PATHS['REPORTS'] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / 'report.json'
CONFIG_PATH   = OUT_DIR / 'config.json'
MANIFEST_PATH = OUT_DIR / 'manifest.json'

SESSIONS_PARQUET  = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'sessions' / 'sessions.parquet'
PAIRS_PARQUET     = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'pairs' / 'pairs.parquet'
USERS_TRAIN_JSON  = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'user_splits' / 'users_train.json'

OUT_SRS_DIR   = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'pairs_srs'
OUT_SRS_DIR.mkdir(parents=True, exist_ok=True)
OUT_PAIRS_SRS = OUT_SRS_DIR / 'pairs.parquet'
OUT_SESSION_SRS = OUT_SRS_DIR / 'session_srs.parquet'

# N_POSSIBLE action types (canonical per CLAUDE.md)
N_POSSIBLE = 6  # load, problem, video, seek, speed, pause

CFG = {'notebook': NOTEBOOK_NAME, 'dataset': DATASET, 'seed': GLOBAL_SEED,
       'srs': {'n_possible_action_types': N_POSSIBLE,
               'cap_percentile': 95,
               'note': 'CAPs computed from training sessions only (no leakage)'}}
write_json_atomic(CONFIG_PATH, CFG)
for path, obj in [(REPORT_PATH, {'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,
                                   'created_at':datetime.now().isoformat(timespec='seconds'),
                                   'metrics':{},'key_findings':[],'sanity_samples':{},'data_fingerprints':{},'notes':[]}),
                   (MANIFEST_PATH, {'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,'artifacts':[]})]:
    write_json_atomic(path, obj)

META = PATHS['META_REGISTRY']
if not META.exists(): write_json_atomic(META, {'schema_version':1,'runs':[]})
m = read_json(META); m['runs'].append({'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,'out_dir':str(OUT_DIR)})
write_json_atomic(META, m)
print(f'[CELL 03b-01] run={RUN_TAG}  done')

[CELL 03b-01] run=20260408_141054  done


In [3]:
# [CELL 03b-02] Load sessions + pairs + train user list

t0 = cell_start('CELL 03b-02', 'Load data')

for p in [SESSIONS_PARQUET, PAIRS_PARQUET]:
    if not p.exists(): raise RuntimeError(f'Missing {p}. Run earlier notebooks first.')

sessions = pd.read_parquet(SESSIONS_PARQUET)
pairs_df  = pd.read_parquet(PAIRS_PARQUET)

print(f'[CELL 03b-02] sessions shape: {sessions.shape}')
print(f'[CELL 03b-02] pairs shape:    {pairs_df.shape}')
print(f'[CELL 03b-02] sessions columns: {list(sessions.columns)}')

# Load train users for CAP computation (no leakage)
if USERS_TRAIN_JSON.exists():
    train_users = set(read_json(USERS_TRAIN_JSON))
    train_sessions = sessions[sessions['user_id'].isin(train_users)]
    print(f'[CELL 03b-02] Train sessions (for CAP): {len(train_sessions):,}')
else:
    # NB04 not yet run — use all sessions (acceptable before split, but note it)
    train_sessions = sessions
    print('[CELL 03b-02] WARNING: users_train.json not found. Using all sessions for CAP (run NB04 to fix).')

cell_end('CELL 03b-02', t0)


[CELL 03b-02] Load data
[CELL 03b-02] sessions shape: (906341, 8)
[CELL 03b-02] pairs shape:    (487696, 7)
[CELL 03b-02] sessions columns: ['session_id', 'user_id', 'course_id', 'start_ts', 'end_ts', 'n_events', 'n_unique_action_types', 'duration_sec']
[CELL 03b-02] WARNING: users_train.json not found. Using all sessions for CAP (run NB04 to fix).
[CELL 03b-02] elapsed=0.59s  done


In [4]:
# [CELL 03b-03] Compute CAPs from training sessions (no leakage)

t0 = cell_start('CELL 03b-03', 'Compute CAPs from training sessions')

CAP_INTENSITY = float(np.percentile(train_sessions['n_events'], 95))
CAP_EXTENT    = float(np.percentile(train_sessions['duration_sec'], 95))

print(f'[CELL 03b-03] CAP_INTENSITY (95th pct n_events):      {CAP_INTENSITY:.1f}')
print(f'[CELL 03b-03] CAP_EXTENT    (95th pct duration_sec):  {CAP_EXTENT:.1f}s ({CAP_EXTENT/60:.1f}min)')
print(f'[CELL 03b-03] N_POSSIBLE action types: {N_POSSIBLE}')

cell_end('CELL 03b-03', t0)


[CELL 03b-03] Compute CAPs from training sessions
[CELL 03b-03] CAP_INTENSITY (95th pct n_events):      115.0
[CELL 03b-03] CAP_EXTENT    (95th pct duration_sec):  6653.0s (110.9min)
[CELL 03b-03] N_POSSIBLE action types: 6
[CELL 03b-03] elapsed=0.01s  done


In [5]:
# [CELL 03b-04] Compute SRS per session (canonical formula)

t0 = cell_start('CELL 03b-04', 'Compute SRS scores')

def compute_srs(n_events, duration_sec, n_unique_action_types):
    """
    SRS = (intensity + extent + composition) / 3
    CAPs computed from training sessions only — no leakage.
    All three components clipped to [0, 1].
    """
    intensity   = min(n_events / CAP_INTENSITY, 1.0)
    extent      = min(duration_sec / CAP_EXTENT, 1.0)
    composition = min(n_unique_action_types / N_POSSIBLE, 1.0)
    return round((intensity + extent + composition) / 3.0, 6)

sessions['srs'] = sessions.apply(
    lambda r: compute_srs(r['n_events'], r['duration_sec'], r['n_unique_action_types']), axis=1
)

srs_vals = sessions['srs']
print(f'[CELL 03b-04] SRS distribution:')
print(f'  mean={srs_vals.mean():.4f}  std={srs_vals.std():.4f}')
print(f'  min={srs_vals.min():.4f}  p25={srs_vals.quantile(.25):.4f}  p50={srs_vals.quantile(.5):.4f}  p75={srs_vals.quantile(.75):.4f}  max={srs_vals.max():.4f}')

assert srs_vals.min() >= 0.0, f'SRS below 0: min={srs_vals.min()}'
assert srs_vals.max() <= 1.0, f'SRS above 1: max={srs_vals.max()}'

low    = (srs_vals < 0.33).mean()
medium = ((srs_vals >= 0.33) & (srs_vals < 0.66)).mean()
high   = (srs_vals >= 0.66).mean()
print(f'\n[CELL 03b-04] Tier breakdown: low={low:.1%}  medium={medium:.1%}  high={high:.1%}')

# Individual components
sessions['srs_intensity']   = (sessions['n_events'] / CAP_INTENSITY).clip(upper=1.0)
sessions['srs_extent']      = (sessions['duration_sec'] / CAP_EXTENT).clip(upper=1.0)
sessions['srs_composition'] = (sessions['n_unique_action_types'] / N_POSSIBLE).clip(upper=1.0)

cell_end('CELL 03b-04', t0, cap_intensity=CAP_INTENSITY, cap_extent=CAP_EXTENT)


[CELL 03b-04] Compute SRS scores
[CELL 03b-04] SRS distribution:
  mean=0.3248  std=0.2325
  min=0.0614  p25=0.1366  p50=0.2456  p75=0.4487  max=1.0000

[CELL 03b-04] Tier breakdown: low=62.8%  medium=25.8%  high=11.4%
[CELL 03b-04] cap_intensity=115.0
[CELL 03b-04] cap_extent=6653.0
[CELL 03b-04] elapsed=6.90s  done


In [6]:
# [CELL 03b-05] Join SRS to pairs

t0 = cell_start('CELL 03b-05', 'Join SRS to pairs')

srs_map = sessions.set_index('session_id')[['srs','srs_intensity','srs_extent','srs_composition']]
pairs_srs = pairs_df.merge(srs_map, on='session_id', how='left')

n_missing = pairs_srs['srs'].isna().sum()
if n_missing > 0:
    print(f'[CELL 03b-05] WARNING: {n_missing} pairs missing SRS (filling with 0)')
    pairs_srs['srs'] = pairs_srs['srs'].fillna(0.0)

print(f'[CELL 03b-05] pairs_srs shape: {pairs_srs.shape}')
print(f'[CELL 03b-05] SRS in pairs: mean={pairs_srs["srs"].mean():.4f}  p50={pairs_srs["srs"].quantile(.5):.4f}')

# Validate SRS in [0,1]
assert (pairs_srs['srs'] >= 0).all() and (pairs_srs['srs'] <= 1).all(), 'SRS out of [0,1]'
print('[CELL 03b-05] All SRS scores in [0, 1]')

cell_end('CELL 03b-05', t0)


[CELL 03b-05] Join SRS to pairs
[CELL 03b-05] pairs_srs shape: (487696, 11)
[CELL 03b-05] SRS in pairs: mean=0.4887  p50=0.4284
[CELL 03b-05] All SRS scores in [0, 1]
[CELL 03b-05] elapsed=0.39s  done


In [11]:
# [CELL 03b-06] Save outputs

t0 = cell_start('CELL 03b-06', 'Save pairs_srs + session_srs')

pairs_srs.to_parquet(OUT_PAIRS_SRS, index=False, compression='zstd')

session_srs_df = sessions[['session_id','user_id','n_events','duration_sec',
                             'n_unique_action_types','srs','srs_intensity','srs_extent','srs_composition']].copy()
session_srs_df.to_parquet(OUT_SESSION_SRS, index=False, compression='zstd')

pairs_srs_sha = sha256_file(OUT_PAIRS_SRS)
sess_srs_sha  = sha256_file(OUT_SESSION_SRS)

print(f'[CELL 03b-06] Saved {OUT_PAIRS_SRS} ({OUT_PAIRS_SRS.stat().st_size/(1<<20):.1f} MB)')
print(f'[CELL 03b-06] Saved {OUT_SESSION_SRS} ({OUT_SESSION_SRS.stat().st_size/(1<<20):.1f} MB)')

# Update report
r = read_json(REPORT_PATH)
r['metrics'] = {
    'n_pairs': len(pairs_srs), 'n_sessions': len(sessions),
    'cap_intensity': CAP_INTENSITY, 'cap_extent': CAP_EXTENT, 'n_possible': N_POSSIBLE,
    'srs_mean': float(srs_vals.mean()), 'srs_p50': float(srs_vals.quantile(.5)),
    'tier_low': float(low), 'tier_medium': float(medium), 'tier_high': float(high),
}
r['data_fingerprints']['pairs_srs'] = {'path': str(OUT_PAIRS_SRS), 'sha256': pairs_srs_sha}
r['data_fingerprints']['session_srs'] = {'path': str(OUT_SESSION_SRS), 'sha256': sess_srs_sha}
write_json_atomic(REPORT_PATH, r)

cell_end('CELL 03b-06', t0)


[CELL 03b-06] Save pairs_srs + session_srs
[CELL 03b-06] Saved /home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/xuetangx/pairs_srs/pairs.parquet (8.9 MB)
[CELL 03b-06] Saved /home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/xuetangx/pairs_srs/session_srs.parquet (12.8 MB)
[CELL 03b-06] elapsed=0.87s  done


## Notebook 03b Complete

SRS computed using canonical formula (CAPs from training sessions only, no leakage).

**Next:** `04_user_split_xuetangx.ipynb` — split users 70/15/15.